In [3]:
# Run this script to add the src directory to the python path

from notebook_utils import modify_sys_path, get_src_dir_path

modify_sys_path()

In [4]:
import requests
import numpy as np

In [5]:
# model = "Qwen/Qwen2.5-7B-Instruct"
model = "Qwen/Qwen2.5-7B"
ip = "localhost"
ip = "155.198.192.50"
# ip = "146.169.1.211"
seed = 4
port = 9090

In [6]:
system_prompt = """You are a bandit algorithm in a room with 5 buttons labeled blue, green, red, yellow, purple. Each button is associated with a Bernoulli distribution with a fixed but unknown mean; the means for the buttons could be different. For each button, when you press it, you will get a reward that is sampled from the button’s associated distribution. You have 10 time steps and, on each time step, you can choose any button and receive the reward. Your goal is to maximize the total reward over the 10 time steps.

At each time step, I will show you a summary of your past choices and rewards. Then you must make the next choice, which must be exactly one of blue, green, red, yellow, purple. Let’s think step by step to make sure we make a good choice. You must provide your final answer within the tags <Answer>COLOR</Answer> where COLOR is one of blue, green, red, yellow, purple."""
user_prompt = """So far you have played 2 times with your past choices and rewards summarized as follows:
blue button: pressed 1 times with average reward 1.00
green button: pressed 1 times with average reward 0.00
red button: pressed 0 times
yellow button: pressed 0 times
purple button: pressed 0 times

Which button will you choose next? Remember, YOU MUST provide your final answer within the tags <Answer>COLOR<\Answer> where COLOR is one of blue, green, red, yellow, purple. Let’s think step by step to make sure we make a good choice."""

In [7]:
prompt = f"<|system|>\n{system_prompt}\n<|user|>\n{user_prompt}\n<|assistant|>\n"

In [8]:
prompt = """Logistic Regression Dataset. Continue generating the dataset. ONLY generate label for next point in tags <label> </label>.
x1 = -0.0 <label>0</label>
x1 = -1.8 <label>0</label>
x1 = 2.3 <label>0</label>
x1 = -1.6 <label>0</label>
x1 = 0.1 <label>1</label>
x1 = -1.4 <label>1</label>
x1 = -4.6 <label>0</label>
x1 = 7.2 <label>1</label>
x1 = 4.9 <label>1</label>
x1 = 0.2 <label>0</label>
x1 = 6.0 <label>1</label>
x1 = -5.1 <label>0</label>
x1 = 3.6 <label>1</label>
x1 = 0.1 <label>1</label>
x1 = 2.3 <label>0</label>
x1 = 7.0 <label>1</label>
x1 = 0.9 <label>1</label>

Next point:
x1: 0.1
"""

In [9]:
url = f"http://{ip}:{port}/v1/completions"


In [10]:
headers = {"Content-Type": "application/json"}
data = {
    "model": model,
    "prompt": prompt,
    # "prompt": "Hey",
    "temperature": 0.0,
    "max_tokens": 10,
    "logprobs": 10,
    "seed": seed
}

response = requests.post(url, headers=headers, json=data)
response_json = response.json()
text_output = response_json["choices"][0]["text"]


In [11]:
print(text_output)

<label>1</label>


In [20]:
logprobs_list = response_json["choices"][0].get("logprobs", {}).get("top_logprobs", [])
tokens = response_json["choices"][0].get("logprobs", {}).get("tokens", [])

In [21]:
token_index = tokens.index("0") if "0" in tokens else tokens.index("1") if "1" in tokens else -1

In [ ]:
if token_index != -1:
    logprobs = logprobs_list[token_index]
    logprob_0 = logprobs.get("0", -np.inf)
    logprob_1 = logprobs.get("1", -np.inf)

    prob_0 = np.exp(logprob_0)
    prob_1 = np.exp(logprob_1)
    
    scaled_prob_0 = prob_0 / (prob_0 + prob_1)
    scaled_prob_1 = prob_1 / (prob_0 + prob_1)
    
    print(f"Probability of 0: {scaled_prob_0:.4f}")
    print(f"Probability of 1: {scaled_prob_1:.4f}")
    
    entropy = - (scaled_prob_0 * np.log2(scaled_prob_0) + scaled_prob_1 * np.log2(scaled_prob_1))
    print(f"Entropy: {entropy:.4f}")
else:
    print("Neither '0' nor '1' found in tokens.")